# Audit the input sources

Start here to inspect the original facility records and the separate public snapshots.
The default `demo` mode uses fictional records. `public` mode reads the existing
supplemental snapshots in this repository. `databricks` mode reads the configured
Unity Catalog tables only when selected inside an authorized Databricks workspace.
Each mode and evidence date has a separate checkpoint directory.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Read and save each input

A source row is a published listing. Several rows can describe the same branch,
and a row can contain contradictory values. The raw facility checkpoint preserves
the source column order for the later `countries` predictor boundary.

In [ ]:
from facility_tool.data import load_records
from facility_tool.profiles import field_profile, phone_coverage, source_profile
from facility_tool.workspace import load_inputs

inputs = load_inputs(MODE, config, globals().get("spark"))
raw_fdr = inputs["facilities"]
records = load_records(ROOT, raw_fdr if len(raw_fdr.columns) else None, include_public=MODE != "demo")
for name, frame in {"records": records, "raw_facilities": raw_fdr,
                    "raw_pins": inputs["pins"], "raw_nfhs": inputs["nfhs"]}.items():
    STORE.write(name, frame)
source_counts = pd.DataFrame(list(records.attrs["source_counts"].items()), columns=["source", "rows_loaded"])
STORE.write("source_counts", source_counts)
print(f"Saved input checkpoints to {STORE.path}")
show(source_counts)

## Coverage and formatting

Nonempty text, plausible coordinates and plausible phone format answer different
questions. A plausible phone format offers a lead for follow-up. Source and URL
coverage count the snapshot rows, including duplicates.

In [ ]:
profiles = {"source_profile": source_profile(records), "phone_coverage": phone_coverage(records),
            "fdr_field_profile": field_profile(raw_fdr)}
for name, frame in profiles.items():
    STORE.write(name, frame)
    print(name.replace("_", " ").title())
    show(frame, limit=60 if name == "fdr_field_profile" else 20)